# PD14 network write example: default model from presets

Assembles the **default Potjans & Diesmann (2014) microcircuit** and writes it to SONATA files with a
`PD14NetworkWriteTask`.

Each of the eight populations `L23E, L23I, L4E, L4I, L5E, L5I, L6E, L6I` is added as a
`PD14PointNeuron` block. Selecting a `preset` fills all population parameters (neuron count,
external indegree, mean rate, layer geometry, initial membrane potential, neuron parameters and the
thalamic connection probability) from the model defaults. The default connection-probability matrix
is added as `PathwayConnectionProbability` blocks.

The network is written to disk only; nothing is registered in the database
(`db_client=None`). For a version that builds the blocks explicitly from the `microcircuit` defaults
and registers the circuit, see `pd14_network_write_example.ipynb`.

In [ ]:
from pathlib import Path

import libsonata
from microcircuit.network_params import default_net_dict

from obi_one.core.info import Info
from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.blocks.point_neuron_populations.pathway import (
    PathwayConnectionProbability,
)
from obi_one.scientific.blocks.point_neuron_populations.pd14 import PD14PointNeuron
from obi_one.scientific.tasks.pd14_network_write import PD14NetworkWriteScanConfig

In [ ]:
output_root = Path("../../../../../../obi-output/pd14_network_write_presets")

# Scaling factors for the number of neurons and the indegrees. 1.0 is the full-scale model
# (about 77,000 neurons and 300 million connections), which needs a lot of memory and disk space;
# smaller values give a scaled-down network for quick tests.
N_SCALING = 0.2
K_SCALING = 0.2

### Add one preset population per PD14 population

`PD14PointNeuron(preset=...)` takes all parameters of that population from the model defaults.
Populations are added first so that each block obtains a reference (`population.ref`) that the
pathway blocks can point to.

In [ ]:
scan_config = PD14NetworkWriteScanConfig.empty_config()
scan_config.set(
    PD14NetworkWriteScanConfig.Initialize(
        n_scaling=N_SCALING,
        k_scaling=K_SCALING,
        psp_ext=float(default_net_dict["PSP_exc_mean"]),
        weight_th_rel_std=float(default_net_dict["weight_rel_std"]),
    ),
    name="initialize",
)
scan_config.set(
    Info(
        campaign_name="PD14 default network",
        campaign_description="Default PD14 model assembled from the population presets",
    ),
    name="info",
)

population_names = list(default_net_dict["populations"])
populations = {name: PD14PointNeuron(preset=name) for name in population_names}
for name, population in populations.items():
    scan_config.add(population, name=name)
population_names

### Add the default connection probabilities

The default `conn_probs` matrix is indexed `conn_probs[target][source]`. One
`PathwayConnectionProbability` block is added for every non-zero entry; zero entries are left out and
have no connections. Each pathway also gets its PSP amplitude from the absolute value of `PSP_matrix_mean` (which includes
the doubled L4E -> L23E pathway) and the default relative weight standard deviation. The thalamic input (902 neurons) is enabled by default, with the
per-population probabilities taken from the presets.

In [ ]:
conn_probs = default_net_dict["conn_probs"]
psp_matrix = default_net_dict["PSP_matrix_mean"]
for i, target_name in enumerate(population_names):
    for j, source_name in enumerate(population_names):
        probability = float(conn_probs[i][j])
        if probability == 0.0:
            continue
        scan_config.add(
            PathwayConnectionProbability(
                source=populations[source_name].ref,
                target=populations[target_name].ref,
                connection_probability=probability,
                psp_mean=abs(float(psp_matrix[i][j])),
                weight_rel_std=float(default_net_dict["weight_rel_std"]),
            ),
            name=f"{source_name}__{target_name}",
        )

scan_config = scan_config.validated_config()

### Generate the single-coordinate scan and execute the task

With `db_client=None` the SONATA files are written to `output_root` and nothing is registered in the
database.

In [ ]:
grid_scan = GridScanGenerationTask(
    form=scan_config,
    output_root=str(output_root),
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()
run_tasks_for_generated_scan(grid_scan, db_client=None)

### Inspect the written SONATA circuit

In [ ]:
instance = grid_scan.single_configs[0]
circuit_root = Path(instance.coordinate_output_root)
print("Written to:", circuit_root)
print("Files:", sorted(p.name for p in circuit_root.iterdir()))

circuit = libsonata.CircuitConfig.from_file(str(circuit_root / "circuit_config.json"))
print("\nNode populations:", sorted(circuit.node_populations))
print("Edge populations:", sorted(circuit.edge_populations))

nodes = circuit.node_population("microcircuit")
print("\nTotal cortical neurons:", nodes.size)
edges = circuit.edge_population("microcircuit__microcircuit__chemical")
print("Total recurrent connections:", edges.size)